# CNN-BiLSTM on RAVDESS + CREMA-D + TESS + SAVEE, whole speakers held out

This notebook runs **one model only**: a CNN-BiLSTM over frozen wav2vec2 (layer 6) frames, trained on all four corpora
(12,162 clips, 121 speakers) and tested on speakers it never heard.

**Protocol (same folds as the SVM / logistic study, Part C).** `StratifiedGroupKFold(5)` over speakers, stratified on
corpus x emotion, seeds 42, 43, 44. About a sixth of the training speakers are validation speakers for early stopping;
the held-out fold is predicted once. Scores are macro-F1 over the labels each set contains, pooled per seed, averaged
over seeds.

**Running it.** Every step is resumable: finished frames and finished (seed, fold) models are skipped. The training
cell takes hours (15 models). Audio must be in place first: see `README.md`, "Run it".

In [1]:
import os, sys, warnings, importlib.util; warnings.filterwarnings("ignore")
from pathlib import Path
# 1. right Python? every package the project needs must be importable in THIS kernel
need = ["torch", "numpy", "pandas", "scipy", "sklearn", "plotly", "optuna", "soundfile", "librosa",
        "transformers", "joblib", "pyarrow", "tqdm", "xgboost"]
missing = [m for m in need if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f"Missing in this kernel: {missing}\nKernel Python: {sys.executable}\n"
                       "Fix: pick the kernel 'Adhoc Tests (.venv)' (Adhoc Tests/.venv/bin/python), or install here with\n"
                       f"  !{sys.executable} -m pip install -r requirements.txt   (then restart the kernel)")
# 2. find the project folder (the one holding multicorpus/) whatever folder the kernel started in
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "multicorpus").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the 'Adhoc Tests' folder (the one containing multicorpus/).")
os.chdir(root); sys.path.insert(0, str(root))
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
import torch, pandas as pd, numpy as np
import plotly.graph_objects as go
from multicorpus import frames, train, LABELS, CORPORA
from multicorpus.corpora import prepare
from unheard_lib import nets
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("kernel:", sys.executable, "\nproject:", root, "| device:", device)

kernel: /Users/rishabsinha/Documents/Projects/AMPBA/Term 3/FP Final Folder/Adhoc Tests/.venv/bin/python 
project: /Users/rishabsinha/Documents/Projects/AMPBA/Term 3/FP Final Folder/Adhoc Tests | device: mps


## 1. Clip table (checks corpus sizes, speakers, and agreement with the reference study)

In [2]:
df = prepare()
df.groupby("source").agg(clips=("filename", "size"), speakers=("speaker_id", "nunique")).loc[CORPORA]

,clips,speakers
source,,
RAVDESS,1440,24
CREMA-D,7442,91
TESS,2800,2
SAVEE,480,4


## 2. wav2vec2 layer-6 frames, one clip at a time, z-scored per speaker (audio only, no labels)

Skipped if the cache exists. First use downloads `facebook/wav2vec2-base` (about 380 MB); about 20 to 60 minutes.

In [3]:
frames.build()

frame cache already built: mc_frames_L6_spknorm.npy


## 3. The model

In [4]:
net = nets.build(train.MODEL, 768, len(LABELS), train.HYPER)
print(train.MODEL, "| hyperparameters:", train.HYPER, "| parameters:", f"{nets.count_parameters(net):,}")
print(net)

cnn_bilstm_w2v | hyperparameters: {'lr': 0.001, 'width': 128, 'dropout': 0.3} | parameters: 530,824
CNNLSTM(
  (proj): FrameProjection(
    (net): Sequential(
      (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (1): Linear(in_features=768, out_features=128, bias=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.3, inplace=False)
    )
  )
  (convs): ModuleList(
    (0-1): 2 x Sequential(
      (0): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ChannelNorm(
        (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      )
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
    )
  )
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (drop): Dropout(p=0.3, inplace=False)
  (head): Linear(in_features=256, out_features=8, bias=True)
)


## 4. Train: 3 seeds x 5 folds (resumable)

In [5]:
train.run(device=device)

training cnn_bilstm_w2v on mps
seed 42 fold 0: test clips 3230, best epoch 17, val macro-F1 0.784, test acc 0.832, 1358s
seed 42 fold 1: test clips 1896, best epoch 15, val macro-F1 0.760, test acc 0.746, 1432s
seed 42 fold 2: test clips 1895, best epoch 8, val macro-F1 0.729, test acc 0.779, 745s
seed 42 fold 3: test clips 3246, best epoch 21, val macro-F1 0.779, test acc 0.742, 1339s
seed 42 fold 4: test clips 1895, best epoch 7, val macro-F1 0.864, test acc 0.734, 681s
seed 43 fold 0: test clips 3230, best epoch 10, val macro-F1 0.734, test acc 0.748, 755s
seed 43 fold 1: test clips 1896, best epoch 19, val macro-F1 0.746, test acc 0.749, 1373s
seed 43 fold 2: test clips 1895, best epoch 11, val macro-F1 0.766, test acc 0.755, 883s
seed 43 fold 3: test clips 3246, best epoch 8, val macro-F1 0.764, test acc 0.857, 718s
seed 43 fold 4: test clips 1895, best epoch 12, val macro-F1 0.818, test acc 0.768, 933s
seed 44 fold 0: test clips 3230, best epoch 10, val macro-F1 0.789, test acc 0

## 5. Score

A seed counts only once all five of its folds are done. Intervals resample speakers; TESS (2 speakers) and SAVEE (4) get none.

In [6]:
res = train.score()

                model              set  clips  speakers  macro_f1_mean_of_seeds  sd_over_seeds    lo    hi  accuracy  seeds
cnn_bilstm|w2v_frames ALL four corpora  12162       121                   0.766          0.003 0.721 0.812     0.773      3
cnn_bilstm|w2v_frames          RAVDESS   1440        24                   0.756          0.004 0.723 0.787     0.761      3
cnn_bilstm|w2v_frames          CREMA-D   7442        91                   0.765          0.006 0.750 0.780     0.764      3
cnn_bilstm|w2v_frames             TESS   2800         2                   0.809          0.010   NaN   NaN     0.814      3
cnn_bilstm|w2v_frames            SAVEE    480         4                   0.691          0.020   NaN   NaN     0.715      3

Macro-F1 (mean of seeds): CNN-BiLSTM vs the earlier pooled-feature models, same folds
                       ALL four corpora  RAVDESS  CREMA-D   TESS  SAVEE
cnn_bilstm|w2v_frames             0.766    0.756    0.765  0.809  0.691
logistic|w2v             

In [7]:
ref = pd.read_csv("reference/summary_partC.csv").set_index("model")
sets = list(res["set"])
fig = go.Figure()
fig.add_bar(name="CNN-BiLSTM", x=sets, y=res["macro_f1_mean_of_seeds"],
            error_y=dict(type="data", symmetric=False, array=(res.hi - res.macro_f1_mean_of_seeds).fillna(0),
                         arrayminus=(res.macro_f1_mean_of_seeds - res.lo).fillna(0)))
for m in ("svm_rbf|w2v", "logistic|w2v"):
    fig.add_bar(name=m, x=sets, y=[ref.loc[m, s] if s in ref.columns else np.nan for s in sets])
fig.update_layout(barmode="group", yaxis_title="macro-F1", yaxis_range=[0, 1], template="simple_white",
                  title="Whole-speaker hold-out, all four corpora pooled")
fig.show()

## 6. Where it errs

In [8]:
pe = pd.read_csv(train.OUT / "cnn_bilstm_per_emotion_f1.csv")
pe.pivot(index="emotion", columns="set", values="f1").reindex(LABELS).dropna(how="all").round(3)

set,ALL four corpora,CREMA-D,RAVDESS,SAVEE,TESS
emotion,,,,,
neutral,0.837,0.819,0.703,0.893,0.900
calm,0.734,NaN,0.739,NaN,NaN
happy,0.742,0.766,0.716,0.633,0.696
sad,0.740,0.714,0.603,0.691,0.891
angry,0.833,0.855,0.874,0.722,0.746
fearful,0.736,0.698,0.778,0.631,0.832
disgust,0.780,0.738,0.852,0.695,0.883
surprised,0.726,NaN,0.788,0.575,0.718


In [9]:
# confusion matrix over all clips, pooled across the scored seeds (row-normalised)
y = df.label_idx.to_numpy(); conf = np.zeros((8, 8))
for seed in train.SEEDS:
    pred = np.full(len(df), -1)
    for fold in range(5):
        f = train.OUT / f"pred_seed{seed}_fold{fold}.npz"
        if f.exists():
            z = np.load(f); pred[z["rows"]] = z["proba"].argmax(1)
    if (pred >= 0).all():
        np.add.at(conf, (y, pred), 1)
conf = conf / conf.sum(1, keepdims=True)
go.Figure(go.Heatmap(z=conf, x=LABELS, y=LABELS, colorscale="Blues", zmin=0, zmax=1)).update_layout(
    xaxis_title="predicted", yaxis_title="true", yaxis_autorange="reversed", template="simple_white",
    title="Row-normalised confusion, all four corpora")

## Reading the result

* The per-corpus rows matter as much as the overall row: CREMA-D is 61% of the clips, and corpus identity is a
  shortcut the model can partly use. This protocol measures new speakers inside corpora already seen, not new corpora.
* The SVM and logistic bars use tuned per-fold settings; this network uses fixed hyperparameters (see README, Limits).
* A difference counts as real only if it exceeds 2 points and the interval excludes zero.